In [5]:
import sys 
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sim.rl import TrackingEnv

In [6]:
from stable_baselines3.common.env_checker import check_env
check_env(TrackingEnv())

Train and chcek the results of the PPO model if not go
 SAC(PPO experimental - 1_000_000, 700_000
 )

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

env = Monitor(TrackingEnv(), "../results/ppo_train")
ppo = PPO("MlpPolicy", env, seed=0, verbose=1)
ppo.learn(total_timesteps=700_000)
ppo.save("../results/ppo_tracking")

Train the SAC model check the results (SAC experimental  - 300_000, 600_000)

In [ ]:
from stable_baselines3 import SAC

env = Monitor(TrackingEnv(), "../results/sac_train")
sac = SAC("MlpPolicy", env, seed=0, verbose=1)
sac.learn(total_timesteps=600_000)
sac.save("../results/sac_tracking")

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for name in ["ppo", "sac"]:
    log = pd.read_csv(f"../results/{name}_train.monitor.csv", skiprows=1)
    ax.plot(np.cumsum(log["l"]), log["r"].rolling(20).mean(), label=name)
ax.set(xlabel="environment steps", ylabel="episode reward (rolling mean)")
ax.legend()    


In [ ]:
from controllers.rl import RlController
from sim.runner import run_tracking, tracking_metrics
from sim.trajectories import circle_ref, figure8_ref, waypoint_ref

runs = [
    ("circle", circle_ref, 40.0),
    ("figure8", figure8_ref, 60.0),
    ("waypoints", lambda t: waypoint_ref(t, hold=8.0), 40.0),
]
rows = []
for algo in ["ppo", "sac"]:
    ctrl = RlController(f"../results/{algo}_tracking", algo)
    for name, ref_fn, duration in runs:
        res = run_tracking(ctrl, ref_fn, duration)
        rows.append({"controller": algo, "trajectory": name, **tracking_metrics(res)})

df = pd.DataFrame(rows)
df.to_csv("../results/rl_nominal.csv", index=False)
df


In [ ]:
combined = pd.concat([pd.read_csv(f"../results/{c}_nominal.csv") for c in ["pid", "mpc", "rl"]])
combined.sort_values(["trajectory", "controller"])


In [ ]:
log = pd.read_csv("../results/sac_train.monitor.csv", skiprows=1)
print(len(log), "episodes,", int(log["l"].sum()), "total steps")


In [ ]:
from sim.rl import TrackingEnvDR
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor

env = Monitor(TrackingEnvDR(), "../results/sac_dr_train")
sac_dr = SAC("MlpPolicy", env, seed=0, verbose=1)
sac_dr.learn(total_timesteps=600_000)
sac_dr.save("../results/sac_dr_tracking")
print("done")

Comparing SAC and DR SAC if DRSAC trained correctly

In [7]:
from controllers.rl import RlController
from sim.runner import run_tracking, tracking_metrics
from sim.trajectories import circle_ref, figure8_ref, waypoint_ref

runs = [
    ("circle", circle_ref, 40.0),
    ("figure8", figure8_ref, 60.0),
    ("waypoints", lambda t: waypoint_ref(t, hold=8.0), 40.0),
]

rows = []
for algo_label, path in [("sac", "../results/sac_tracking"), ("sac_dr", "../results/sac_dr_tracking")]:
    ctrl = RlController(path, "sac")
    for name, ref_fn, dur in runs:
        res = run_tracking(ctrl, ref_fn, dur)
        rows.append({"controller": algo_label, "trajectory": name, **tracking_metrics(res)})

import pandas as pd
pd.DataFrame(rows)

,controller,trajectory,pos_rmse,heading_rmse,effort,compute_ms_mean,compute_ms_max
0,sac,circle,0.504887,0.051507,15001.458097,0.128695,22.293917
1,sac,figure8,0.477332,0.458775,61359.633126,0.119226,0.570500
2,sac,waypoints,0.826827,0.219795,70339.601549,0.115687,0.259250
3,sac_dr,circle,0.620358,0.298409,26095.134024,0.115928,0.219792
4,sac_dr,figure8,0.354789,0.490558,75905.649285,0.116078,0.217708
5,sac_dr,waypoints,0.856260,0.111426,51141.880670,0.115503,0.202250
